# Initial Data Review

This notebook provides a reproducible structural summary of the raw wind-turbine dataset before EDA, preprocessing, and modelling.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

DATA_PATH = Path.cwd().parent / 'data' / 'raw' / 'wind_turbine_detection.csv'
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / 'PROJECT 1' / 'data' / 'raw' / 'wind_turbine_detection.csv'

df = pd.read_csv(DATA_PATH)

display(Markdown('## Data shape'))
display(pd.DataFrame({'rows': [df.shape[0]], 'columns': [df.shape[1]]}))

display(Markdown('## Data dictionary by category'))
metadata = {
    'timestamp': ('Identifiers & Metadata', 'Date and time of the 10-minute SCADA record.'),
    'turbine_id': ('Identifiers & Metadata', 'Unique identifier assigned to each wind turbine.'),
    'rated_power_kW': ('Environmental Conditions', 'Rated maximum power output capacity of the turbine.'),
    'wind_speed_mps': ('Environmental Conditions', 'Wind speed measured in meters per second.'),
    'wind_direction_deg': ('Environmental Conditions', 'Direction from which the wind is blowing, measured in degrees.'),
    'turbulence_intensity': ('Environmental Conditions', 'Degree of fluctuation in wind speed relative to its average value.'),
    'air_density_kgm3': ('Environmental Conditions', 'Air density in kilograms per cubic meter, which influences energy production.'),
    'ambient_temp_C': ('Environmental Conditions', 'Ambient air temperature surrounding the turbine.'),
    'humidity_pct': ('Environmental Conditions', 'Relative humidity of the surrounding air expressed as a percentage.'),
    'power_output_kW': ('Operational Control & State', 'Actual electrical power generated by the turbine.'),
    'rotor_speed_rpm': ('Operational Control & State', 'Rotational speed of the turbine rotor in revolutions per minute.'),
    'generator_speed_rpm': ('Operational Control & State', 'Rotational speed of the generator shaft in revolutions per minute.'),
    'blade_pitch_angle_deg': ('Operational Control & State', 'Blade pitch angle used to regulate rotor speed and power generation.'),
    'yaw_misalignment_deg': ('Operational Control & State', 'Angular difference between the turbine orientation and incoming wind direction.'),
    'gearbox_oil_temp_C': ('Thermal Metrics (Component Health)', 'Temperature of the gearbox lubricating oil.'),
    'gearbox_bearing_temp_C': ('Thermal Metrics (Component Health)', 'Temperature of the gearbox bearings.'),
    'generator_bearing_temp_C': ('Thermal Metrics (Component Health)', 'Temperature of the generator bearings.'),
    'generator_winding_temp_C': ('Thermal Metrics (Component Health)', 'Temperature of the generator windings.'),
    'main_bearing_temp_C': ('Thermal Metrics (Component Health)', 'Temperature of the turbine main bearing.'),
    'nacelle_temp_C': ('Thermal Metrics (Component Health)', 'Internal temperature inside the turbine nacelle.'),
    'drivetrain_vibration_rms_mmps': ('Vibration & Frequency Diagnostics', 'Overall drivetrain vibration level measured as RMS vibration velocity.'),
    'tower_vibration_mmps': ('Vibration & Frequency Diagnostics', 'Vibration level of the turbine tower.'),
    'vib_fft_bearing_bpfo': ('Vibration & Frequency Diagnostics', 'FFT vibration amplitude at the bearing outer-race defect frequency (BPFO).'),
    'vib_fft_bearing_bpfi': ('Vibration & Frequency Diagnostics', 'FFT vibration amplitude at the bearing inner-race defect frequency (BPFI).'),
    'vib_fft_gearmesh': ('Vibration & Frequency Diagnostics', 'FFT vibration amplitude at the gearbox gear mesh frequency.'),
    'vib_fft_sideband': ('Vibration & Frequency Diagnostics', 'FFT vibration amplitude at sideband frequencies, often associated with localized mechanical faults.'),
    'oil_particle_count': ('Lubrication System', 'Number of wear particles detected in the lubricating oil.'),
    'oil_pressure_bar': ('Lubrication System', 'Lubrication system oil pressure measured in bar.'),
    'operating_hours_total': ('Asset Lifecycle & Maintenance', 'Total accumulated operating hours of the turbine.'),
    'cumulative_energy_MWh': ('Asset Lifecycle & Maintenance', 'Total lifetime energy generated by the turbine in megawatt-hours.'),
    'load_cycles': ('Asset Lifecycle & Maintenance', 'Number of load cycles experienced by the turbine components.'),
    'hours_since_last_maintenance': ('Asset Lifecycle & Maintenance', 'Operating hours elapsed since the last maintenance activity.'),
    'prior_fault_count': ('Asset Lifecycle & Maintenance', 'Total number of previously recorded fault events.'),
    'component_age_days': ('Asset Lifecycle & Maintenance', 'Age of the monitored component in days.'),
    'failure': ('Target Variable', 'Binary target variable indicating drivetrain condition (0 = Normal operation, 1 = Fault detected).'),
}
column_summary = pd.DataFrame({
    'column': df.columns,
    'data_type': df.dtypes.astype(str).values,
    'category': [metadata.get(c, ('Unspecified', ''))[0] for c in df.columns],
    'description': [metadata.get(c, ('Unspecified', 'Description not provided.'))[1] for c in df.columns],
    'missing_values': df.isna().sum().values,
    'unique_values': df.nunique(dropna=False).values,
})
for category, category_table in column_summary.groupby('category', sort=False):
    display(Markdown(f'### {category}'))
    display(category_table.drop(columns='category').reset_index(drop=True))

display(Markdown('## Numerical value ranges'))
numerical_columns = df.select_dtypes(include='number').columns
numerical_ranges = df[numerical_columns].agg(['min', 'max']).T.rename(columns={'min': 'minimum', 'max': 'maximum'})
display(numerical_ranges)

display(Markdown('## Categorical columns and unique values'))
categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns
for column in categorical_columns:
    values = df[column].dropna().unique().tolist()
    preview = values if len(values) <= 20 else values[:20] + ['...']
    display(pd.DataFrame({'column': [column], 'unique_count': [len(values)], 'unique_values': [preview]}))

display(Markdown('## Notable observations from the initial review'))
missing = df.isna().sum()
missing_columns = missing[missing.gt(0)]
failure_rate = df['failure'].mean() if 'failure' in df else None
observations = [
    f'- The dataset contains **{df.shape[0]:,} rows** and **{df.shape[1]} columns**.',
    f'- There are **{df.duplicated().sum():,} exact duplicate rows**.',
    f'- Missing values occur in **{len(missing_columns)} columns**: ' + (', '.join(f'`{col}` ({count:,})' for col, count in missing_columns.items()) if len(missing_columns) else 'none') + '.',
    f'- `timestamp` has **{df["timestamp"].nunique():,} unique values** and `turbine_id` has **{df["turbine_id"].nunique():,} unique values**; parsing time and accounting for turbine-level structure will be important.',
    f'- The target `failure` is imbalanced: **{failure_rate:.2%}** positive records.' if failure_rate is not None else '- No `failure` target column was found.',
    '- Several low-cardinality numerical fields, such as `rated_power_kW`, may be treated as categorical or ordinal during feature engineering.',
]
display(Markdown('\n'.join(observations)))

## Data shape

,rows,columns
0,131760,35


## Data dictionary by category

### Identifiers & Metadata

,column,data_type,description,missing_values,unique_values
0,timestamp,str,Date and time of the 10-minute SCADA record.,0,8784
1,turbine_id,str,Unique identifier assigned to each wind turbine.,0,15


...

## Categorical columns and unique values

,column,unique_count,unique_values
0,timestamp,8784,"[1/1/2024 0:00, 1/1/2024 0:10, 1/1/2024 0:20, ..."


,column,unique_count,unique_values
0,turbine_id,15,"[T001, T002, T003, T004, T005, T006, T007, T00..."


## Notable observations from the initial review

- The dataset contains **131,760 rows** and **35 columns**.
- There are **0 exact duplicate rows**.
- Missing values occur in **3 columns**: `gearbox_oil_temp_C` (645), `generator_bearing_temp_C` (690), `oil_pressure_bar` (636).
- `timestamp` has **8,784 unique values** and `turbine_id` has **15 unique values**; parsing time and accounting for turbine-level structure will be important.
- The target `failure` is imbalanced: **3.00%** positive records.
- Several low-cardinality numerical fields, such as `rated_power_kW`, may be treated as categorical or ordinal during feature engineering.

In [2]:
# Drivetrain-condition sensor profile
drivetrain_sensors = [
    'gearbox_oil_temp_C', 'gearbox_bearing_temp_C', 'generator_bearing_temp_C',
    'generator_winding_temp_C', 'main_bearing_temp_C', 'nacelle_temp_C',
    'drivetrain_vibration_rms_mmps', 'tower_vibration_mmps',
    'vib_fft_bearing_bpfo', 'vib_fft_bearing_bpfi', 'vib_fft_gearmesh',
    'vib_fft_sideband', 'oil_particle_count', 'oil_pressure_bar'
]
available_sensors = [column for column in drivetrain_sensors if column in df.columns]

display(Markdown('## Drivetrain-condition sensor summary'))
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
sensor_summary = df[available_sensors].describe(percentiles=percentiles).T.rename(columns={
    'min': 'minimum', 'max': 'maximum', 'mean': 'mean', '50%': 'median',
    'std': 'standard_deviation', '1%': 'p01', '5%': 'p05', '25%': 'p25',
    '75%': 'p75', '95%': 'p95', '99%': 'p99'
})
sensor_summary = sensor_summary[['minimum', 'maximum', 'mean', 'median', 'standard_deviation', 'p01', 'p05', 'p25', 'p75', 'p95', 'p99']]

# Rule-based flags point to distributions for review; they are not fault thresholds.
relative_spread = (sensor_summary['p99'] - sensor_summary['p01']) / sensor_summary['median'].abs().clip(lower=1e-9)
upper_tail_ratio = sensor_summary['p99'] / sensor_summary['median'].abs().clip(lower=1e-9)
lower_tail_ratio = sensor_summary['p01'].abs() / sensor_summary['median'].abs().clip(lower=1e-9)
flags = pd.Series('', index=sensor_summary.index, dtype='object')
flags.loc[relative_spread > 5] += 'wide central range; '
flags.loc[upper_tail_ratio > 3] += 'high-end extremes; '
flags.loc[(sensor_summary['p99'] - sensor_summary['p95']) > 2 * (sensor_summary['p75'] - sensor_summary['p25']).clip(lower=1e-9)] += 'long upper tail; '
flags.loc[(sensor_summary['p01'] - sensor_summary['minimum']).abs() > 2 * (sensor_summary['p75'] - sensor_summary['p25']).clip(lower=1e-9)] += 'low-end extremes; '
sensor_summary['investigation_flag'] = flags.str.rstrip('; ').replace('', 'No rule-based flag')
display(sensor_summary)

flagged = sensor_summary.loc[sensor_summary['investigation_flag'].ne('No rule-based flag'), 'investigation_flag']
display(Markdown('### Variables to investigate'))
if flagged.empty:
    display(Markdown('No sensor met the rule-based screening criteria.'))
else:
    display(Markdown('\n'.join(f'- `{sensor}`: {reason}.' for sensor, reason in flagged.items())))

display(Markdown('> These flags identify candidates for EDA (for example, by turbine, operating state, and failure label). They do not by themselves indicate abnormal operating thresholds.'))

## Drivetrain-condition sensor summary

,minimum,maximum,mean,median,standard_deviation,p01,p05,p25,p75,p95,p99,investigation_flag
gearbox_oil_temp_C,35.814,109.988,58.205181,54.3070,11.835815,41.86014,44.32400,49.4275,65.3635,81.29660,85.27372,No rule-based flag
gearbox_bearing_temp_C,37.901,113.862,62.713522,58.1350,13.312458,44.50659,47.24700,52.7840,70.8585,88.52100,93.08041,No rule-based flag
generator_bearing_temp_C,41.375,111.263,62.394198,58.4060,11.634394,46.52707,49.03235,53.8000,69.1320,85.31265,88.97224,No rule-based flag
generator_winding_temp_C,44.771,158.996,78.593421,66.1890,25.953997,50.74918,53.83000,59.4890,92.9305,129.73105,133.44782,No rule-based flag
main_bearing_temp_C,30.311,94.146,52.409791,49.0420,10.960906,36.91159,39.32300,44.2780,58.8500,74.07100,77.79500,No rule-based flag
nacelle_temp_C,-2.681,44.331,19.791835,19.8255,9.254256,2.31959,5.26895,11.9820,27.5150,34.10800,37.70241,No rule-based flag
drivetrain_vibration_rms_mmps,1.216,5.770,1.834684,1.6920,0.515269,1.29400,1.35600,1.5060,1.9760,2.99400,3.91000,No rule-based flag
tower_vibration_mmps,0.801,3.717,1.134653,1.0790,0.243325,0.82800,0.86100,0.9570,1.2650,1.54500,2.03941,No rule-based flag
vib_fft_bearing_bpfo,0.301,2.577,0.434847,0.3880,0.235574,0.30700,0.32000,0.3490,0.4410,0.58800,1.89700,high-end extremes; long upper tail
vib_fft_bearing_bpfi,0.301,2.846,0.421709,0.3840,0.198516,0.30700,0.32000,0.3490,0.4380,0.56300,1.48400,high-end extremes; long upper tail


### Variables to investigate

- `vib_fft_bearing_bpfo`: high-end extremes; long upper tail.
- `vib_fft_bearing_bpfi`: high-end extremes; long upper tail.
- `vib_fft_gearmesh`: high-end extremes; long upper tail.
- `vib_fft_sideband`: high-end extremes; long upper tail.
- `oil_particle_count`: high-end extremes; long upper tail.
- `oil_pressure_bar`: low-end extremes.

> These flags identify candidates for EDA (for example, by turbine, operating state, and failure label). They do not by themselves indicate abnormal operating thresholds.